# TP – IQL para recomendación (Netflix sintético)

Notebook único. **Ejecutarlo desde `2026/tp`** (donde está la carpeta `protocolo/`). Los datos se cargan por nombre de dataset (`DATASET = "netflix"`) con `protocolo.datos`; no hay rutas de archivos en el notebook.

1. Configuración, datos y utilidades de evaluación
2. Modelo (codificador de estado, Q, V, π, red objetivo)
3. Pérdidas y entrenamiento
4. Tests
5. Logs y primer entrenamiento con diagnóstico
6. Búsqueda de τ, β, γ en validación
7. Evaluación final en test, con 3 semillas de entrenamiento

## 1. Configuración, datos y utilidades de evaluación

In [1]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from protocolo.datos import Episodes, load_test, load_train, make_episodes, num_items, split_by_user
from protocolo.evaluar import baseline_factories, evaluate
from protocolo.simulador import SessionConfig

DATASET = "netflix"
SEED = 0
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUTPUT = Path(f"outputs/dt/{DATASET}/seed{SEED}")
print("device:", DEVICE)

device: cpu


In [2]:
def seen_before_mask(items, n_items):
    """(B, T, n_items): True si el ítem ya se mostró antes del paso t en el episodio."""
    one_hot = F.one_hot(items, num_classes=n_items)
    seen = one_hot.cumsum(dim=1) > 0
    return torch.cat([torch.zeros_like(seen[:, :1]), seen[:, :-1]], dim=1)

In [3]:
train_all = load_train(DATASET)
train, val = split_by_user(train_all)          # 90 % ajuste / 10 % validación, estratificado, semilla 0
N_ITEMS = num_items(DATASET)
train_ep = make_episodes(train, length=20)
val_ep = make_episodes(val, length=20)
print("usuarios train:", len(train), "| val:", len(val), "| n_items:", N_ITEMS)
print("episodios train:", train_ep.items.shape, "| val:", val_ep.items.shape)
print("retorno medio de un episodio de train:", float(train_ep.ratings.sum(1).mean()), "(el protocolo dice ≈ 64,3)")

usuarios train: 14400 | val: 1600 | n_items: 752
episodios train: (73827, 20) | val: (8275, 20)
retorno medio de un episodio de train: 64.29808877510938 (el protocolo dice ≈ 64,3)


In [4]:
N_USERS_VAL = 400
SEEDS_VAL = (0,)
N_SEEDS_TEST = (0, 1, 2)

def correr(policy, users, group_visible=True, warmup=0, seeds=(0,)):
    """Corre el protocolo y devuelve métricas promedio (primero por usuario, como metricas.summarize)."""
    config = SessionConfig(horizon=20, warmup=warmup, group_visible=group_visible)
    df = evaluate(policy, users, N_ITEMS, config, seeds)
    return df.groupby("user_id")[["normalized", "return", "hit_rate"]].mean().mean()

rng = np.random.default_rng(0)
idx = np.sort(rng.choice(len(val), size=min(N_USERS_VAL, len(val)), replace=False))
val_sub = [val[i] for i in idx]

## 2. Modelo IQL: decisiones de diseño

Decisiones de diseño (las pide la guía; hay que declararlas en el informe):

- **Estado:** transformer causal sobre los pares (ítem, rating) ya revelados, más el grupo (o "desconocido") y la posición. Devuelve **T+1 estados** `h_0 … h_T`: `h_t` es lo que se sabe antes de recomendar `a_t` y no ve `a_t` ni `r_t`.
- **Q, V y π tienen cada una su propio codificador** (no comparten parámetros). Es más caro, pero garantiza que cada pérdida solo actualiza su red. Hay un test que lo verifica.
- **Q objetivo:** copia de Q actualizada con promedio de Polyak (λ = 0,005).
- **Recompensa centrada:** `(r − 3) / 2`. No cambia qué ítem es mejor, pero fija la escala de Q, V y de las ventajas, y con ella el efecto de β.
- **Un solo Q** (sin double-Q), como en la guía.
- **Grupo oculto:** en el 30 % de los episodios de train el grupo se reemplaza por "desconocido", igual que en el DT.
- **Checkpoint:** se guarda el modelo de la **última epoch**, no el de menor pérdida. Las pérdidas de IQL no miden la calidad de la política (la guía lo remarca).
- **Políticas:** `pi` ordena por las logits de π; `q` ordena por Q sobre todos los ítems disponibles.

Requiere haber ejecutado las secciones 2 (modelo del DT, por `seen_before_mask`) y las utilidades de evaluación.

In [5]:
import copy

class StateEncoder(nn.Module):
    """Transformer causal sobre el historial de la sesión.

    Entrada: items (B, L), ratings (B, L), groups (B,), con L entre 0 y horizon.
    Salida: (B, L+1, d). La posición k ve los pares (a_j, r_j) con j < k, el grupo y la posición.
    """

    def __init__(self, n_items, horizon=20, d_model=64, n_layers=2, n_heads=4, n_groups=8, dropout=0.1):
        super().__init__()
        if d_model % n_heads:
            raise ValueError("d_model debe ser divisible por n_heads")
        self.horizon = horizon
        self.n_groups = n_groups
        self.item_emb = nn.Embedding(n_items, d_model)
        self.rating_emb = nn.Linear(1, d_model)
        self.group_emb = nn.Embedding(n_groups + 1, d_model)   # +1: grupo desconocido
        self.pos_emb = nn.Embedding(horizon + 1, d_model)
        self.start = nn.Parameter(torch.zeros(d_model))        # token del historial vacío (estado s_0)
        layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=4 * d_model, dropout=dropout,
            activation="gelu", norm_first=True, batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers, enable_nested_tensor=False)
        self.ln_out = nn.LayerNorm(d_model)
        for emb in (self.item_emb, self.group_emb, self.pos_emb):
            nn.init.normal_(emb.weight, std=0.02)

    def forward(self, items, ratings, groups):
        B, L = items.shape
        if L > self.horizon:
            raise ValueError(f"L={L} supera el horizonte {self.horizon}")
        dtype = self.item_emb.weight.dtype
        pairs = self.item_emb(items) + self.rating_emb(((ratings.to(dtype) - 3.0) / 2.0).unsqueeze(-1))
        x = torch.cat([self.start.expand(B, 1, -1), pairs], dim=1)             # (B, L+1, d)
        pos = self.pos_emb(torch.arange(L + 1, device=items.device)).unsqueeze(0)
        x = x + pos + self.group_emb(groups).unsqueeze(1)
        causal = torch.triu(torch.ones(L + 1, L + 1, dtype=torch.bool, device=items.device), diagonal=1)
        return self.ln_out(self.encoder(x, mask=causal))


class IQLNetwork(nn.Module):
    """Codificador de estado + cabeza lineal. out_dim = n_items (Q, π) o 1 (V)."""

    def __init__(self, out_dim, **enc_kwargs):
        super().__init__()
        self.encoder = StateEncoder(**enc_kwargs)
        self.head = nn.Linear(enc_kwargs.get("d_model", 64), out_dim)

    def forward(self, items, ratings, groups):
        return self.head(self.encoder(items, ratings, groups))      # (B, L+1, out_dim)


class IQL(nn.Module):
    def __init__(self, n_items, horizon=20, d_model=64, n_layers=2, n_heads=4, n_groups=8, dropout=0.1):
        super().__init__()
        self.n_items, self.horizon, self.n_groups = n_items, horizon, n_groups
        kw = dict(n_items=n_items, horizon=horizon, d_model=d_model, n_layers=n_layers,
                  n_heads=n_heads, n_groups=n_groups, dropout=dropout)
        self.q = IQLNetwork(n_items, **kw)
        self.v = IQLNetwork(1, **kw)
        self.pi = IQLNetwork(n_items, **kw)
        self.q_target = copy.deepcopy(self.q)
        for p in self.q_target.parameters():
            p.requires_grad_(False)

    def train(self, mode=True):
        super().train(mode)
        self.q_target.eval()            # la red objetivo nunca usa dropout
        return self

    @torch.no_grad()
    def update_target(self, lam=0.005):
        for pt, p in zip(self.q_target.parameters(), self.q.parameters()):
            pt.mul_(1.0 - lam).add_(p, alpha=lam)

## 3. Pérdidas y entrenamiento

In [6]:
def expectile_loss(diff, tau):
    """diff = Q_obj - V. Con tau > 0,5 penaliza más quedarse por debajo (diff > 0)."""
    weight = tau * (diff > 0) + (1.0 - tau) * (diff <= 0)
    return (weight * diff.pow(2)).mean()


def reward_scale(ratings, center=True):
    return (ratings - 3.0) / 2.0 if center else ratings


def td_target(r, v_all, gamma):
    """r: (B, T). v_all: (B, T+1) con V(s_0..s_T). El último paso es terminal: el objetivo es sólo r."""
    not_done = torch.ones_like(r)
    not_done[:, -1] = 0.0
    return r + gamma * not_done * v_all[:, 1:].detach()


def iql_losses(model, batch, gamma, tau, beta, max_weight=100.0, center=True):
    items, ratings, groups = batch["items"], batch["ratings"], batch["groups"]
    B, T = items.shape
    r = reward_scale(ratings, center)
    a = items.unsqueeze(-1)

    with torch.no_grad():                                   # Q objetivo en la acción registrada
        q_obj = model.q_target(items, ratings, groups)[:, :T].gather(-1, a).squeeze(-1)

    v_all = model.v(items, ratings, groups).squeeze(-1)     # (B, T+1)
    loss_v = expectile_loss(q_obj - v_all[:, :T], tau)      # solo actualiza V

    q = model.q(items, ratings, groups)[:, :T].gather(-1, a).squeeze(-1)
    loss_q = F.mse_loss(q, td_target(r, v_all, gamma))      # solo actualiza Q (V entra sin gradiente)

    with torch.no_grad():                                   # pesos AWR sin gradiente, truncados
        weights = torch.exp(beta * (q_obj - v_all[:, :T])).clamp(max=max_weight)
    logits = model.pi(items, ratings, groups)[:, :T]
    logits = logits.masked_fill(seen_before_mask(items, model.n_items), float("-inf"))
    ce = F.cross_entropy(logits.reshape(-1, model.n_items), items.reshape(-1), reduction="none").reshape(B, T)
    loss_pi = (weights * ce).mean()                         # solo actualiza π

    return {"v": loss_v, "q": loss_q, "pi": loss_pi, "weights": weights}


def episodes_to_iql_tensors(ep, device="cpu"):
    return {
        "items": torch.as_tensor(ep.items, dtype=torch.long, device=device),
        "ratings": torch.as_tensor(ep.ratings, dtype=torch.float32, device=device),
        "groups": torch.as_tensor(ep.groups, dtype=torch.long, device=device),
    }


@torch.no_grad()
def iql_mean_losses(model, data, batch_size=512, **loss_kw):
    was_training = model.training
    model.eval()
    n = data["items"].shape[0]
    tot = {"v": 0.0, "q": 0.0, "pi": 0.0, "w": 0.0}
    for start in range(0, n, batch_size):
        batch = {k: v[start:start + batch_size] for k, v in data.items()}
        l = iql_losses(model, batch, **loss_kw)
        nb = batch["items"].shape[0]
        for k in ("v", "q", "pi"):
            tot[k] += l[k].item() * nb
        tot["w"] += l["weights"].mean().item() * nb
    if was_training:
        model.train()
    return {k: x / n for k, x in tot.items()}


def train_iql(train_ep, val_ep, n_items, seed=0, epochs=10, batch_size=256, lr=3e-4, weight_decay=0.01,
              gamma=0.9, tau=0.9, beta=3.0, max_weight=100.0, center=True, target_lambda=0.005,
              group_dropout=0.3, grad_clip=1.0, d_model=64, n_layers=2, n_heads=4, dropout=0.1,
              max_batches=None, device="cpu", output=None, verbose=True):
    torch.manual_seed(seed)
    generator = torch.Generator().manual_seed(seed)

    horizon = int(train_ep.items.shape[1])
    train_data = episodes_to_iql_tensors(train_ep, device)
    val_data = episodes_to_iql_tensors(val_ep, device)

    model = IQL(n_items=n_items, horizon=horizon, d_model=d_model, n_layers=n_layers,
                n_heads=n_heads, dropout=dropout).to(device)
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                                  lr=lr, weight_decay=weight_decay)
    n_episodes = train_data["items"].shape[0]
    steps_per_epoch = max(1, (n_episodes + batch_size - 1) // batch_size)
    if max_batches is not None:
        steps_per_epoch = min(steps_per_epoch, max_batches)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=lr, total_steps=max(1, epochs * steps_per_epoch))

    loss_kw = dict(gamma=gamma, tau=tau, beta=beta, max_weight=max_weight, center=center)
    config = dict(n_items=n_items, horizon=horizon, d_model=d_model, n_layers=n_layers, n_heads=n_heads,
                  dropout=dropout, seed=seed, epochs=epochs, batch_size=batch_size, lr=lr,
                  weight_decay=weight_decay, group_dropout=group_dropout, target_lambda=target_lambda, **loss_kw)

    history = []
    for epoch in range(1, epochs + 1):
        model.train()
        order = torch.randperm(n_episodes, generator=generator).to(device)
        acc = {"v": 0.0, "q": 0.0, "pi": 0.0, "w": 0.0, "clip": 0.0}
        batches = 0

        for step in range(steps_per_epoch):
            index = order[step * batch_size:(step + 1) * batch_size]
            if index.numel() == 0:
                continue
            batch = {k: v[index] for k, v in train_data.items()}
            if group_dropout > 0:
                drop = torch.rand(batch["groups"].shape, generator=generator).to(device) < group_dropout
                batch["groups"] = torch.where(drop, torch.full_like(batch["groups"], model.n_groups), batch["groups"])

            losses = iql_losses(model, batch, **loss_kw)
            loss = losses["v"] + losses["q"] + losses["pi"]     # cada pérdida llega sólo a su red
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()
            scheduler.step()
            model.update_target(target_lambda)

            w = losses["weights"]
            for k in ("v", "q", "pi"):
                acc[k] += losses[k].item()
            acc["w"] += w.mean().item()
            acc["clip"] += (w >= max_weight).float().mean().item()
            batches += 1

        n_b = max(1, batches)
        val = iql_mean_losses(model, val_data, **loss_kw)
        history.append({"epoch": epoch,
                        "train_v": acc["v"] / n_b, "train_q": acc["q"] / n_b, "train_pi": acc["pi"] / n_b,
                        "w_medio": acc["w"] / n_b, "w_truncado": acc["clip"] / n_b,
                        "val_v": val["v"], "val_q": val["q"], "val_pi": val["pi"]})
        if verbose:
            h = history[-1]
            print(f"epoch {epoch:3d}/{epochs} | V {h['train_v']:.4f} Q {h['train_q']:.4f} π {h['train_pi']:.4f} | "
                  f"peso medio {h['w_medio']:.2f} (truncado {h['w_truncado']:.1%}) | val π {h['val_pi']:.4f}")

    model.eval()                      # se conserva el modelo final (no se elige por pérdida)
    if output is not None:
        out = Path(output)
        out.mkdir(parents=True, exist_ok=True)
        torch.save({"state_dict": model.state_dict(), "config": config}, out / "modelo_iql.pt")
        pd.DataFrame(history).to_csv(out / "historial.csv", index=False)
        (out / "config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")
        if verbose:
            print(f"checkpoint e historial en {out}")
    return model, pd.DataFrame(history)


def load_iql(path, device="cpu"):
    ckpt = torch.load(path, map_location=device, weights_only=False)
    c = ckpt["config"]
    model = IQL(n_items=c["n_items"], horizon=c["horizon"], d_model=c["d_model"], n_layers=c["n_layers"],
                n_heads=c["n_heads"], dropout=c["dropout"]).to(device)
    model.load_state_dict(ckpt["state_dict"])
    return model.eval()


class IQLPolicy:
    """mode="pi": ordena por las logits de π. mode="q": ordena por Q sobre los ítems disponibles."""

    def __init__(self, model, mode="pi"):
        if mode not in ("pi", "q"):
            raise ValueError("mode debe ser 'pi' o 'q'")
        self.model = model.eval()
        self.mode = mode
        self.group = model.n_groups

    def reset(self, group, horizon):
        self.group = self.model.n_groups if group is None else int(group)

    @torch.no_grad()
    def scores(self, items_seen, ratings_seen):
        items_seen = np.asarray(items_seen, dtype=np.int64)
        ratings_seen = np.asarray(ratings_seen, dtype=np.float32)
        t = len(items_seen)
        device = next(self.model.parameters()).device
        items = torch.as_tensor(items_seen, device=device).unsqueeze(0)       # (1, t)
        ratings = torch.as_tensor(ratings_seen, device=device).unsqueeze(0)
        groups = torch.tensor([self.group], device=device)
        net = self.model.pi if self.mode == "pi" else self.model.q
        out = net(items, ratings, groups)[0, t].clone()                       # estado s_t
        out[items_seen] = float("-inf")
        return out.cpu().numpy()

## 4. Tests del IQL
Equivalentes a los del DT: no fuga, uso del historial, expectil, paso terminal, aislamiento de gradientes, red objetivo y política.

In [7]:
def make_iql_batch(B=4, T=20, n_items=50, n_groups=8, seed=0):
    g = torch.Generator().manual_seed(seed)
    items = torch.stack([torch.randperm(n_items, generator=g)[:T] for _ in range(B)])
    ratings = torch.randint(1, 6, (B, T), generator=g).float()
    groups = torch.randint(0, n_groups, (B,), generator=g)
    return {"items": items, "ratings": ratings, "groups": groups}


def make_small_iql(n_items=50):
    torch.manual_seed(0)
    return IQL(n_items=n_items, horizon=20, d_model=32, n_layers=1, n_heads=2, dropout=0.0)


def run_iql_tests():
    model, b = make_small_iql(), make_iql_batch()
    items, ratings, groups = b["items"], b["ratings"], b["groups"]

    # formas: T+1 estados
    for net, dim in ((model.q, 50), (model.v, 1), (model.pi, 50)):
        assert net(items, ratings, groups).shape == (4, 21, dim), "forma de salida"

    # no fuga: el estado s_t no depende de a_t, r_t ni de lo posterior
    model.eval()
    with torch.no_grad():
        for t in (0, 7, 19):
            base = model.q(items, ratings, groups)[:, t]
            items2 = items.clone();   items2[:, t:] = torch.randint(0, 50, items2[:, t:].shape)
            ratings2 = ratings.clone(); ratings2[:, t:] = torch.randint(1, 6, ratings2[:, t:].shape).float()
            out = model.q(items2, ratings2, groups)[:, t]
            assert torch.allclose(base, out, atol=1e-5), f"FUGA en t={t}"
        # y sí usa el pasado y el grupo
        base = model.pi(items, ratings, groups)[:, 7]
        r3 = ratings.clone(); r3[:, 6] += 2.0
        assert not torch.allclose(base, model.pi(items, r3, groups)[:, 7]), "no usa el rating previo"
        i3 = items.clone(); i3[:, 6] = (i3[:, 6] + 1) % 50
        assert not torch.allclose(base, model.pi(i3, ratings, groups)[:, 7]), "no usa el ítem previo"
        assert not torch.allclose(base, model.pi(items, ratings, (groups + 1) % 8)[:, 7]), "no usa el grupo"

    # expectil: el gradiente se anula en el expectil de [1, 2, 3, 10] (tabla de la guía)
    u = torch.tensor([1.0, 2.0, 3.0, 10.0])
    for tau, v_star in ((0.5, 4.0), (0.7, 5.5), (0.9, 8.0)):
        v = torch.tensor(v_star, requires_grad=True)
        expectile_loss(u - v, tau).backward()
        assert abs(v.grad.item()) < 1e-4, f"expectil incorrecto para tau={tau}"

    # objetivo TD: el último paso es terminal y gamma=0 deja solo la recompensa
    r = torch.ones(2, 20)
    v_all = torch.randn(2, 21)
    t1 = td_target(r, v_all, 0.9)
    v2 = v_all.clone(); v2[:, -1] += 100.0
    assert torch.allclose(t1, td_target(r, v2, 0.9)), "el último paso no debe usar V(s')"
    assert torch.allclose(t1[:, -1], r[:, -1])
    assert torch.allclose(t1[:, 0], r[:, 0] + 0.9 * v_all[:, 1])
    assert torch.allclose(td_target(r, v_all, 0.0), r)

    # aislamiento de gradientes: cada pérdida sólo llega a su red; Q objetivo no recibe gradiente
    nets = {"v": model.v, "q": model.q, "pi": model.pi, "q_target": model.q_target}
    model.train()
    for key in ("v", "q", "pi"):
        model.zero_grad(set_to_none=True)
        iql_losses(model, b, gamma=0.9, tau=0.9, beta=3.0)[key].backward()
        for name, net in nets.items():
            has = any(p.grad is not None and p.grad.abs().sum() > 0 for p in net.parameters())
            assert has == (name == key), f"la pérdida {key} llegó a la red {name}"

    # red objetivo: promedio de Polyak
    m = make_small_iql()
    before = next(m.q_target.parameters()).clone()
    with torch.no_grad():
        for p in m.q.parameters():
            p.add_(1.0)
    m.update_target(0.1)
    assert torch.allclose(next(m.q_target.parameters()), before + 0.1, atol=1e-6)

    # política: forma, máscara de ítems vistos y coherencia con el modelo
    pm = make_small_iql(n_items=40)
    for mode in ("pi", "q"):
        pol = IQLPolicy(pm, mode)
        pol.reset(group=None, horizon=20)
        s = pol.scores(np.array([], dtype=np.int64), np.array([], dtype=np.float64))
        assert s.shape == (40,) and np.isfinite(s).all()
        seen = np.array([1, 2, 3])
        pol.reset(group=3, horizon=20)
        s = pol.scores(seen, np.array([4.0, 5.0, 3.0]))
        assert s.shape == (40,) and np.isneginf(s[seen]).all()
        assert np.isfinite(s[~np.isin(np.arange(40), seen)]).all()
        full_items = torch.tensor([[1, 2, 3] + list(range(5, 22))])
        full_ratings = torch.tensor([[4.0, 5.0, 3.0] + [3.0] * 17])
        net = pm.pi if mode == "pi" else pm.q
        with torch.no_grad():
            ref = net(full_items, full_ratings, torch.tensor([3]))[0, 3].numpy()
        ok = ~np.isin(np.arange(40), seen)
        assert np.allclose(s[ok], ref[ok], atol=1e-5), f"la política {mode} no coincide con el modelo"
    print("OK: todos los tests del IQL pasaron")

run_iql_tests()

OK: todos los tests del IQL pasaron


## 5. Logs y primer entrenamiento

`LOGS` elige con qué logs se entrena: `"random"` (los originales), `"mixto"` o `"experto"`. Las líneas base se ajustan con **los mismos logs** que el modelo, como pide el protocolo. 

El primer entrenamiento usa valores por defecto (τ = 0,9, β = 3, γ = 0,9) y sirve para mirar las curvas y el **peso medio de AWR**:
- si casi todos los pesos están truncados, β es demasiado grande para la escala de las ventajas;
- si todos valen ≈ 1, π es casi behavior cloning.

In [8]:
LOGS = "random"        # "random" | "mixto" | "experto"
IQL_EPOCHS = 10

if LOGS == "random":
    iql_train_ep, iql_val_ep = train_ep, val_ep
    train_baseline = train
else:
    from protocolo.logs import logged_episodes, logs_cache
    _tr = logged_episodes(LOGS, train, train, N_ITEMS, 20, seed=0, cache=logs_cache(DATASET, LOGS, "train"))
    _va = logged_episodes(LOGS, val, train, N_ITEMS, 20, seed=1, cache=logs_cache(DATASET, LOGS, "val"))
    iql_train_ep, iql_val_ep = _tr.episodes, _va.episodes
    train_baseline = _tr.logged_users()

print(f"logs: {LOGS} | episodios train {iql_train_ep.items.shape} | val {iql_val_ep.items.shape}")
print("retorno medio de un episodio de train:", float(iql_train_ep.ratings.sum(1).mean()))

logs: random | episodios train (73827, 20) | val (8275, 20)
retorno medio de un episodio de train: 64.29808877510938


In [9]:
# Prueba rápida (1 epoch, 5 batches)
_ = train_iql(iql_train_ep, iql_val_ep, N_ITEMS, seed=SEED, epochs=1, max_batches=5, device=DEVICE)

epoch   1/1 | V 0.1037 Q 1.1717 π 7.6889 | peso medio 1.14 (truncado 0.0%) | val π 10.2099


In [ ]:
iql_model, iql_hist = train_iql(iql_train_ep, iql_val_ep, N_ITEMS, seed=SEED, epochs=IQL_EPOCHS,
                                tau=0.9, beta=3.0, gamma=0.9, device=DEVICE)

fig, ax = plt.subplots(1, 3, figsize=(13, 3.4))
for k, lab in (("train_v", "V"), ("train_q", "Q")):
    ax[0].plot(iql_hist.epoch, iql_hist[k], label=lab)
ax[0].set_title("pérdidas V y Q (train)"); ax[0].legend()
ax[1].plot(iql_hist.epoch, iql_hist.train_pi, label="train"); ax[1].plot(iql_hist.epoch, iql_hist.val_pi, label="val")
ax[1].set_title("π ponderada (AWR)"); ax[1].legend()
ax[2].plot(iql_hist.epoch, iql_hist.w_medio, label="peso medio"); ax[2].plot(iql_hist.epoch, iql_hist.w_truncado, label="fracción truncada")
ax[2].set_title("pesos AWR"); ax[2].legend()
for a in ax: a.set_xlabel("epoch")
plt.tight_layout(); plt.show()

In [ ]:
# Primer chequeo con el protocolo (validación) frente a las líneas base ajustadas con los mismos logs
refs_iql = baseline_factories(train_baseline, N_ITEMS)
filas = []
for visible in (True, False):
    for nombre in ("item_mean", "group_mean", "group_posterior"):
        if nombre in refs_iql:
            m = correr(refs_iql[nombre](), val_sub, group_visible=visible, seeds=SEEDS_VAL)
            filas.append({"policy": nombre, "group_visible": visible, "normalized": m["normalized"]})
    for mode in ("pi", "q"):
        m = correr(IQLPolicy(iql_model, mode), val_sub, group_visible=visible, seeds=SEEDS_VAL)
        filas.append({"policy": f"IQL_{mode}", "group_visible": visible, "normalized": m["normalized"]})
pd.DataFrame(filas).pivot_table(index="policy", columns="group_visible", values="normalized").round(3)

## 6. Búsqueda de τ, β y γ en validación

Se elige con el **protocolo** (`normalized` en validación, grupo visible y oculto promediados), no con las pérdidas. La grilla de la guía son τ ∈ {0,7; 0,9}, β ∈ {3; 10} y γ ∈ {0; 0,9}: 8 entrenamientos, y cada uno se evalúa con `pi` y con `q`.

γ = 0 convierte el problema en un bandit contextual (Q solo predice la recompensa inmediata). Si γ > 0 no mejora, es porque en este generador el rating no depende de lo mostrado antes: no hay valor futuro que aprender.

In [ ]:
GRID = [dict(tau=t, beta=b, gamma=g) for t in (0.7, 0.9) for b in (3.0, 10.0) for g in (0.0, 0.9)]

res = []
for cfg in GRID:
    m_i, _ = train_iql(iql_train_ep, iql_val_ep, N_ITEMS, seed=SEED, epochs=IQL_EPOCHS,
                       device=DEVICE, verbose=False, **cfg)
    for mode in ("pi", "q"):
        for visible in (True, False):
            m = correr(IQLPolicy(m_i, mode), val_sub, group_visible=visible, seeds=SEEDS_VAL)
            res.append({**cfg, "mode": mode, "group_visible": visible, **m.to_dict()})
    ult = [r for r in res if r["tau"] == cfg["tau"] and r["beta"] == cfg["beta"] and r["gamma"] == cfg["gamma"]]
    print(cfg, {md_: round(np.mean([r["normalized"] for r in ult if r["mode"] == md_]), 3) for md_ in ("pi", "q")})

grid_iql = pd.DataFrame(res)
resumen_grid = grid_iql.groupby(["tau", "beta", "gamma", "mode"])["normalized"].mean().unstack("mode").round(3)
resumen_grid

In [ ]:
mejor_cfg = {}
for mode in ("pi", "q"):
    s = grid_iql[grid_iql["mode"] == mode].groupby(["tau", "beta", "gamma"])["normalized"].mean()
    tau_, beta_, gamma_ = s.idxmax()
    mejor_cfg[mode] = dict(tau=float(tau_), beta=float(beta_), gamma=float(gamma_))
    print(mode, "->", mejor_cfg[mode], f"(normalized val {s.max():.3f})")

## 7. Evaluación final en test (**una sola vez**)

Para cada modo (`pi`, `q`) se reentrena la mejor configuración con **3 semillas de entrenamiento** y se evalúa en test con 3 semillas de sesión, warm-up 0 y 5, grupo visible y oculto. Se reporta la media y el desvío **entre semillas de entrenamiento**; una mejora menor a ese desvío no es una mejora.

Costo: 2 modos × 3 semillas × 4 condiciones × usuarios × 3 semillas de sesión × 20 pasos. Para probar primero, poné `N_USERS_TEST_IQL = 200`; la tabla final tiene que salir con todos.

In [ ]:
TRAIN_SEEDS = (0, 1, 2)
N_USERS_TEST_IQL = None   # None = todos los usuarios de test

test_users_iql = load_test(DATASET)
if N_USERS_TEST_IQL:
    test_users_iql = test_users_iql[:N_USERS_TEST_IQL]

filas = []
for mode in ("pi", "q"):
    for s in TRAIN_SEEDS:
        m_s, _ = train_iql(iql_train_ep, iql_val_ep, N_ITEMS, seed=s, epochs=IQL_EPOCHS, device=DEVICE,
                           verbose=False, output=f"outputs/iql/{DATASET}/{LOGS}/{mode}/seed{s}", **mejor_cfg[mode])
        for warmup in (0, 5):
            for visible in (True, False):
                m = correr(IQLPolicy(m_s, mode), test_users_iql, group_visible=visible,
                           warmup=warmup, seeds=N_SEEDS_TEST)
                filas.append({"policy": f"IQL_{mode}", "train_seed": s, "warmup": warmup,
                              "group_visible": visible, **m.to_dict()})
tabla_iql = pd.DataFrame(filas)

por_semilla = tabla_iql.pivot_table(index=["policy", "train_seed"], columns=["warmup", "group_visible"], values="normalized")
resumen_iql = pd.DataFrame({
    "media": tabla_iql.groupby(["policy", "warmup", "group_visible"])["normalized"].mean(),
    "desvío entre semillas": tabla_iql.groupby(["policy", "warmup", "group_visible"])["normalized"].std(),
}).round(3)
resumen_iql.unstack(["warmup", "group_visible"])

In [ ]:
# Líneas base en test, ajustadas con los mismos logs que el IQL (la tabla mínima del reporte)
base_test = {k: v for k, v in baseline_factories(train_baseline, N_ITEMS).items()
             if k in ("random", "popularity", "item_mean", "group_mean", "group_posterior")}
filas = []
for nombre, make in base_test.items():
    for warmup in (0, 5):
        for visible in (True, False):
            m = correr(make(), test_users_iql, group_visible=visible, warmup=warmup, seeds=N_SEEDS_TEST)
            filas.append({"policy": nombre, "warmup": warmup, "group_visible": visible, "normalized": m["normalized"]})
tabla_base = pd.DataFrame(filas)

iql_medio = tabla_iql.groupby(["policy", "warmup", "group_visible"])["normalized"].mean().reset_index()
comparativa = pd.concat([tabla_base, iql_medio])
comparativa.pivot_table(index="policy", columns=["warmup", "group_visible"], values="normalized").round(3)